# **Sesión 15: Limpieza y Transformación de Datos: Datos Temporales**
### Fechas, horas y duraciones con `datetime` y pandas

---

| | |
|---|---|
| **Materia** | Programación para Analítica Descriptiva y Predictiva |
| **Programa** | Maestría en Inteligencia Artificial y Analítica de Datos (MIAAD) — UACJ |
| **Unidad** | 02 Análisis Descriptivo de los Datos |
| **Tema** | Limpieza y transformación de datos: datos temporales |
| **Entorno** | Google Colab |

---

## Objetivo

Representar, convertir, limpiar y transformar datos temporales (fechas, horas y duraciones) con el módulo `datetime` de Python y con pandas, para usarlos correctamente en el análisis descriptivo.

## Agenda

| # | Sección | Contenido |
|---|---|---|
| 1 | Introducción | Qué son los datos temporales, tipos y problemas frecuentes |
| 2 | Módulo `datetime` | `date`, `time`, `datetime` y sus componentes |
| 3 | Zonas horarias | Fechas *naive* y *aware*, UTC, horario de verano |
| 4 | Marca de tiempo Unix | `timestamp()` y `fromtimestamp()` |
| 5 | Texto ↔ fecha | `strptime()`, `strftime()` y códigos de formato |
| 6 | Aritmética temporal | Restas y `timedelta` |
| 7 | Datos temporales en pandas | `parse_dates`, `pd.to_datetime()`, `format` |
| 8 | Componentes con `.dt` | Año, mes, día de la semana, hora, trimestre, periodos |
| 9 | Operaciones y filtrado | Duraciones, `pd.Timedelta`, `pd.to_timedelta()`, filtros por fecha |
| 10 | Limpieza de fechas | Formatos mixtos, `NaT`, `errors='coerce'`, ambigüedad día/mes |
| 11 | Zonas horarias en pandas | `tz_localize()`, `tz_convert()` |
| 12 | Caso real 1 | Calidad del aire, 2020 (Airdata): validación de la línea de tiempo, patrones y fecha como índice |
| 13 | Caso real 2 | Catálogo de Netflix: fechas en texto con espacios, `Period` y anomalías |
| 14 | Caso real 3 | Calidad del aire en Europa (OpenAQ): fechas en UTC, varias estaciones y muestreo irregular |
| 15 | Cierre | Resumen y buenas prácticas |

## Videos de la sesión

| # | Video | Duración | Secciones de este notebook |
|---|---|---|---|
| 1 | [Introducción a los datos temporales](https://www.youtube.com/watch?v=4jzeLXkEEIc) | 9 min | 1 |
| 2 | [Instrucciones básicas de `datetime`](https://youtu.be/tNaUe71hy8E) | 23 min | 2 a 6 |
| 3 | [Datos temporales en un DataFrame](https://youtu.be/YP_FetX_xbg) | 15 min | 7 a 10 |

El notebook es autocontenido: cada tema incluye su explicación y ejemplos ejecutables. Las secciones 11 a 14 amplían los videos con casos más avanzados y datos reales.

## Archivos requeridos

Descarga de Moodle los archivos `temporal01.csv`, `temporal02.csv` y `Airdata.csv` y súbelos a la carpeta **`MyDrive/Unidad02/`** de tu Google Drive. En la sección 7 conectaremos Colab con Drive para leerlos.

Los datos de las secciones 13 y 14 se descargan directamente de repositorios públicos en GitHub; no requieren ningún archivo.

## Cómo usar este notebook

Ejecuta las celdas en orden (`Shift + Enter`). Antes de algunas celdas encontrarás una **Pregunta**: piensa qué resultado esperas, ejecuta y compara. La celda de **Observación** que sigue explica el resultado.

Las celdas marcadas con ⚠️ producen un **error intencional** para mostrar fallas frecuentes; cada una va seguida de su explicación.

## 1. Introducción a los datos temporales

### 1.1 ¿Qué son?

Los **datos temporales** representan el tiempo: fechas, horas o combinaciones de ambas. Permiten analizar eventos en función del tiempo, detectar tendencias y patrones (por hora, día de la semana, mes), calcular duraciones y agrupar por periodos.

Formatos básicos:

| Tipo | Ejemplo |
|---|---|
| Fecha (*date*) | `2025-02-28` |
| Hora (*time*) | `18:00:00` |
| Fecha y hora (*datetime*) | `2025-02-28 18:00:00` |

### 1.2 Dos formas de usar una variable temporal

1. **Como eje principal del análisis**: se estudia cómo cambia otra variable a lo largo del tiempo. Es el campo de las **series temporales**; las fechas suelen ser únicas y se usan como índice.
2. **Como una variable más**: la fecha es una columna de la que se extrae información (mes, día de la semana, duración) para describir o filtrar los datos.

La diferencia se refleja en cómo se guarda la fecha en el `DataFrame`:

| | Fecha como **columna** | Fecha como **índice** |
|---|---|---|
| Representa | Un atributo más del registro | El identificador de cada fila |
| Valores repetidos | Permitidos | Deben ser únicos |
| Extraer partes (mes, año, día) | Directo con `.dt` | Con el índice (`df.index.month`) |
| Operaciones aritméticas entre fechas | Directas | Menos cómodas |
| Filtrar por rango | Con máscaras (`between`) | Con `.loc['2020-03']`, más rápido en datos grandes |
| Cambiar la frecuencia, ventanas deslizantes | No directamente | `resample()`, `rolling()` |
| Uso principal | Análisis descriptivo | Series temporales y pronóstico (*forecasting*) |

En esta sesión nos enfocamos en el **punto 2** (columna), que es el que se usa en el análisis descriptivo. En la sección 12.7 veremos una breve comparación con la fecha como índice.

### 1.3 Tipos de datos temporales en una tabla

Considera la siguiente agenda:

| Fecha | Hora | Fecha y hora | Marca de tiempo (Unix, UTC) | Actividad |
|---|---|---|---|---|
| 28-02-2025 | 08:00 | 28-02-2025 08:00 | 1740729600 | Desayuno |
| 28-02-2025 | 09:00 | 28-02-2025 09:00 | 1740733200 | Reunión con asesor |
| 28-02-2025 | 10:00 | 28-02-2025 10:00 | 1740736800 | Clase de maestría |
| 28-02-2025 | 11:00 | 28-02-2025 11:00 | 1740740400 | Clase de maestría |
| 28-02-2025 | 12:00 | 28-02-2025 12:00 | 1740744000 | Clase de maestría |
| 28/02/2025 | 13:00 | 28-02-2025 13:00 | 1740747600 | Comida |
| 28022025 | 14:00 | 28-02-2025 14:00 | 1740751200 | Curso de Python |

En ella aparecen:

| Concepto | Qué representa | En la tabla |
|---|---|---|
| **date, time, datetime** | Fecha, hora, o ambas | Columnas *Fecha*, *Hora* y *Fecha y hora* |
| **Marca de tiempo** (*timestamp*) | El instante en que ocurre un evento; puede expresarse como número de segundos desde 1970 (sección 4) | Cada fila: "el desayuno ocurrió el 28-02-2025 a las 08:00" = `1740729600` |
| **Duración** (*timedelta*) | Distancia entre dos marcas de tiempo | La clase de maestría: de 10:00 a 12:00 → 2 horas |
| **Otros formatos** | La misma fecha escrita de otra forma | `28/02/2025` y `28022025` en la columna *Fecha* |

### 1.4 ¿Por qué requieren limpieza?

En los archivos reales, las fechas casi nunca llegan listas para usarse:

| Problema | Ejemplo |
|---|---|
| Se leen como **texto** | `'2025-02-28'` es un `str`: no se puede restar ni extraer el mes |
| **Formatos mezclados** en una columna | `2025/03/01`, `2025-03-02 14:45`, `March 3 2025` |
| **Ambigüedad** día/mes | `01/03/2025`: ¿1 de marzo o 3 de enero? |
| **Fechas imposibles** | `2025-02-30` |
| **Faltantes** | Celdas vacías |
| **Zonas horarias** | La misma hora registrada en UTC y en hora local |

Al final de esta sesión sabrás detectar y resolver cada uno.

## 2. El módulo `datetime` de Python

Python incluye el módulo `datetime` en su biblioteca estándar. Sus clases principales:

| Clase | Construcción | Representa |
|---|---|---|
| `date` | `date(año, mes, día)` | Una fecha |
| `time` | `time(hora, minuto, segundo, microsegundo)` | Una hora del día |
| `datetime` | `datetime(año, mes, día, hora, minuto, segundo, microsegundo)` | Fecha y hora |
| `timedelta` | `timedelta(days=, hours=, minutes=, weeks=, ...)` | Una duración |
| `timezone` | `timezone.utc`, `timezone(timedelta(hours=-6))` | Un desfase fijo respecto a UTC |

Hay dos formas comunes de importarlo:

```python
# Con alias: acceso a todas las clases como dt.date, dt.datetime, dt.timedelta
import datetime as dt

# Importando solo lo necesario
from datetime import datetime, date, timedelta
```

Ambas son válidas. En este notebook usaremos la primera: deja claro de dónde viene cada clase y evita confundir el **módulo** `datetime` con la **clase** `datetime`.

In [ ]:
# Importamos el módulo con el alias dt
import datetime as dt

# Fecha: año, mes, día
print('Fecha       :', dt.date(2025, 2, 28))

# Hora: hora, minuto, segundo
print('Hora        :', dt.time(18, 12, 25))

# Fecha y hora
print('Fecha y hora:', dt.datetime(2025, 2, 28, 18, 12, 25))

Python **valida** las fechas al construirlas.

**Pregunta**: ¿qué ocurre si intentamos crear el 30 de febrero?

> ⚠️ La siguiente celda produce un **error intencional**.

In [ ]:
# ERROR INTENCIONAL: febrero no tiene 30 días
fecha_invalida = dt.date(2025, 2, 30)

**Explicación del error**: `ValueError: day is out of range for month`.

`date` no acepta fechas que no existen. Esto es útil: una fecha imposible en los datos es un **error de captura** que se debe detectar. En pandas, como veremos en la sección 10, estas fechas pueden convertirse en faltantes (`NaT`) en lugar de detener el programa.

### 2.1 Componentes de una fecha

Una vez creado el objeto, se accede a sus partes mediante **atributos** (sin paréntesis) y **métodos** (con paréntesis):

| Componente | Devuelve | Tipo |
|---|---|---|
| `fecha.year` | Año | atributo |
| `fecha.month` | Mes (1–12) | atributo |
| `fecha.day` | Día del mes | atributo |
| `fecha.weekday()` | Día de la semana: **lunes = 0**, …, domingo = 6 | método |
| `fecha.isoweekday()` | Día de la semana ISO: **lunes = 1**, …, domingo = 7 | método |

**Pregunta**: el 28 de febrero de 2025 fue viernes. ¿Qué devolverán `weekday()` e `isoweekday()`?

In [ ]:
fecha = dt.date(2025, 2, 28)

print('Año                 :', fecha.year)
print('Mes                 :', fecha.month)
print('Día                 :', fecha.day)
print('Día de la semana    :', fecha.weekday())      # lunes = 0
print('Día de la semana ISO:', fecha.isoweekday())   # lunes = 1

**Observación**: viernes es `4` con `weekday()` y `5` con `isoweekday()`. Confundir ambas numeraciones es un error frecuente al agrupar por día de la semana; verifica siempre cuál estás usando.

### 2.2 Otras funciones de `date`

| Función | Descripción |
|---|---|
| `dt.date.today()` | Fecha actual del sistema |
| `dt.datetime.now()` | Fecha y hora actuales del sistema |
| `dt.date.fromisoformat('AAAA-MM-DD')` | Convierte un texto en formato ISO a `date` |
| `fecha.replace(year=, month=, day=)` | Devuelve una **copia** con los componentes indicados cambiados |
| `fecha.toordinal()` | Número de días transcurridos desde el 1 de enero del año 1 |

In [ ]:
# Fecha y fecha-hora actuales (el resultado depende del momento de ejecución)
print('Hoy            :', dt.date.today())
print('Ahora          :', dt.datetime.now())

# Texto en formato ISO (AAAA-MM-DD) → date
fecha = dt.date.fromisoformat('2025-03-01')
print('Desde texto ISO:', fecha)

# replace() no modifica 'fecha': devuelve una nueva
nueva_fecha = fecha.replace(year=2028)
print('Año cambiado   :', nueva_fecha)
print('Original       :', fecha)

# Días desde el 01-01-0001
print('Ordinal        :', fecha.toordinal())

> En Colab, `today()` y `now()` usan el reloj del servidor, que está en **UTC**. Por eso la hora puede no coincidir con la de Ciudad Juárez. La sección 3 explica cómo manejarlo.

### 2.3 Componentes de una hora (`time`)

| Atributo | Devuelve |
|---|---|
| `t.hour` | Hora (0–23) |
| `t.minute` | Minuto |
| `t.second` | Segundo |
| `t.microsecond` | Microsegundo |

In [ ]:
# Hora con microsegundos: 14:30:15.000023
mi_hora = dt.time(14, 30, 15, 23)

print('Hora completa:', mi_hora)
print('Hora         :', mi_hora.hour)
print('Minuto       :', mi_hora.minute)
print('Segundo      :', mi_hora.second)
print('Microsegundo :', mi_hora.microsecond)

# isoformat() convierte el objeto en texto con formato estándar
print('Como texto   :', mi_hora.isoformat(), type(mi_hora.isoformat()))

### 2.4 Componentes de `datetime`

`datetime` combina los componentes de `date` y de `time`. Además, se puede separar en sus dos partes con `.date()` y `.time()`.

In [ ]:
fecha_hora = dt.datetime(2024, 3, 1, 14, 30, 45, 123456)

# Componentes de fecha
print('Año         :', fecha_hora.year)
print('Mes         :', fecha_hora.month)
print('Día         :', fecha_hora.day)
print('Día semana  :', fecha_hora.weekday(), '(viernes)')

# Componentes de hora
print('Hora        :', fecha_hora.hour)
print('Minuto      :', fecha_hora.minute)
print('Segundo     :', fecha_hora.second)
print('Microsegundo:', fecha_hora.microsecond)

# Separar en fecha y hora
print('Solo fecha  :', fecha_hora.date())
print('Solo hora   :', fecha_hora.time())

## 3. Zonas horarias

Un `datetime` puede ser:

- **Naive** (ingenuo): no tiene zona horaria. `tzinfo` es `None`. "14:30" sin saber de dónde.
- **Aware** (consciente): tiene zona horaria. "14:30 en UTC" o "14:30 en Ciudad Juárez".

**¿Por qué importa?** Si un sistema registra en UTC y otro en hora local, la misma venta aparecerá con horas distintas; al unir los datos, las duraciones y los agrupamientos por hora quedan mal.

**UTC** (Tiempo Universal Coordinado) es la referencia mundial: las demás zonas se expresan como un desfase respecto a UTC (por ejemplo, UTC−6).

In [ ]:
# Fecha sin zona horaria (naive)
fecha_naive = dt.datetime(2024, 3, 1, 14, 30)
print('Naive :', fecha_naive, '| tzinfo:', fecha_naive.tzinfo)

# Fecha con zona horaria UTC (aware)
fecha_utc = dt.datetime(2024, 3, 1, 14, 30, tzinfo=dt.timezone.utc)
print('Aware :', fecha_utc, '| tzinfo:', fecha_utc.tzinfo)

**Observación**: la fecha *aware* se imprime con su desfase: `+00:00` indica UTC.

### 3.1 Asignar o convertir una zona horaria

Son dos operaciones distintas:

| Operación | Método | Efecto |
|---|---|---|
| **Asignar** | `fecha.replace(tzinfo=zona)` | Declara en qué zona está la hora. **No cambia** la hora. |
| **Convertir** | `fecha.astimezone(zona)` | Expresa el mismo instante en otra zona. **Cambia** la hora. |

Para zonas reales con horario de verano se usa `ZoneInfo`, de la biblioteca estándar, con el nombre oficial de la zona (base de datos IANA), por ejemplo `'America/Ciudad_Juarez'`.

In [ ]:
from zoneinfo import ZoneInfo

juarez = ZoneInfo('America/Ciudad_Juarez')

# 1) ASIGNAR: declaramos que las 14:30 son hora UTC (la hora no cambia)
evento_utc = dt.datetime(2025, 3, 1, 14, 30).replace(tzinfo=dt.timezone.utc)
print('Evento en UTC    :', evento_utc)

# 2) CONVERTIR: el mismo instante expresado en Ciudad Juárez (la hora cambia)
evento_juarez = evento_utc.astimezone(juarez)
print('Mismo instante en Juárez:', evento_juarez)

**Pregunta**: Ciudad Juárez aplica horario de verano (sincronizado con Estados Unidos). ¿El desfase respecto a UTC será el mismo en enero que en julio?

In [ ]:
invierno = dt.datetime(2025, 1, 15, 12, 0, tzinfo=juarez)
verano = dt.datetime(2025, 7, 15, 12, 0, tzinfo=juarez)

# strftime('%z') muestra el desfase respecto a UTC como texto (+HHMM / -HHMM)
print('Enero:', invierno, '| desfase:', invierno.strftime('%z'))
print('Julio:', verano, '| desfase:', verano.strftime('%z'))

**Observación**: en invierno Juárez está en UTC−7 y en verano en UTC−6. Un desfase fijo (`timezone(timedelta(hours=-6))`) sería correcto solo la mitad del año; `ZoneInfo` aplica el cambio automáticamente.

## 4. Marca de tiempo Unix (*Unix timestamp*)

Muchos sistemas (bases de datos, registros de servidores, APIs) guardan el tiempo como un **número**: los segundos transcurridos desde la **época Unix**, el `1970-01-01 00:00:00 UTC`. Por ejemplo, `1326244364`.

| Operación | Código |
|---|---|
| Número → fecha | `dt.datetime.fromtimestamp(numero, tz=dt.timezone.utc)` |
| Fecha → número | `fecha.timestamp()` |

> Si no se indica `tz`, `fromtimestamp()` usa la zona horaria del equipo, y `timestamp()` supone que una fecha *naive* está en hora local. Para resultados reproducibles, trabaja en UTC.

In [ ]:
# Número de segundos desde 1970-01-01 UTC → datetime en UTC
instante = dt.datetime.fromtimestamp(1326244364, tz=dt.timezone.utc)
print('Fecha y hora UTC:', instante)

# datetime aware → número de segundos
fecha = dt.datetime(2025, 2, 28, 18, 0, tzinfo=dt.timezone.utc)
print('Timestamp       :', fecha.timestamp())

## 5. Convertir entre texto y fecha

Los datos casi siempre llegan como **texto**. Dos métodos hacen la conversión en ambos sentidos:

| Método | Dirección | Mnemotecnia |
|---|---|---|
| `dt.datetime.strptime(texto, formato)` | texto → fecha | **p**: *parse* (interpretar) |
| `fecha.strftime(formato)` | fecha → texto | **f**: *format* (dar formato) |

El **formato** se describe con códigos que empiezan con `%`:

| Código | Significado | Ejemplo |
|---|---|---|
| `%Y` | Año con 4 dígitos | `2025` |
| `%y` | Año con 2 dígitos | `25` |
| `%m` | Mes con 2 dígitos | `02` |
| `%B` / `%b` | Nombre del mes / abreviado (en inglés) | `February` / `Feb` |
| `%d` | Día del mes | `28` |
| `%A` / `%a` | Día de la semana / abreviado (en inglés) | `Friday` / `Fri` |
| `%H` | Hora (00–23) | `18` |
| `%I` / `%p` | Hora (01–12) / AM o PM | `06` / `PM` |
| `%M` | Minuto | `05` |
| `%S` | Segundo | `09` |
| `%j` | Día del año (001–366) | `059` |
| `%z` | Desfase respecto a UTC | `-0600` |

Lista completa: [documentación de Python — códigos de formato](https://docs.python.org/3/library/datetime.html#format-codes).

Todos los caracteres que no son códigos (guiones, diagonales, espacios, dos puntos) deben coincidir **exactamente** con el texto.

In [ ]:
# Texto en formato AAAA-MM-DD
texto = '2025-02-27'
fecha = dt.datetime.strptime(texto, '%Y-%m-%d')
print(fecha, type(fecha))

# Texto con mes abreviado en inglés: 27-Feb-2025
fecha = dt.datetime.strptime('27-Feb-2025', '%d-%b-%Y')
print(fecha)

# Texto con hora en formato de 12 horas: 02/27/2025 06:30 PM
fecha = dt.datetime.strptime('02/27/2025 06:30 PM', '%m/%d/%Y %I:%M %p')
print(fecha)

**Pregunta**: ¿qué ocurre si el formato no coincide con el texto?

> ⚠️ La siguiente celda produce un **error intencional**.

In [ ]:
# ERROR INTENCIONAL: el texto usa '/' pero el formato indica '-'
fecha = dt.datetime.strptime('27/02/2025', '%d-%m-%Y')

**Explicación del error**: `ValueError: time data '27/02/2025' does not match format '%d-%m-%Y'`.

`strptime()` es estricto: cada separador debe coincidir. Corrección: `dt.datetime.strptime('27/02/2025', '%d/%m/%Y')`.

### 5.1 De fecha a texto: `strftime()`

Útil para presentar fechas en reportes o para crear etiquetas (por ejemplo, `'2025-02'` para agrupar por mes).

In [ ]:
fecha = dt.datetime(2025, 2, 28, 18, 5)

print(fecha.strftime('%d/%m/%Y'))            # formato común en México
print(fecha.strftime('%Y-%m'))               # etiqueta año-mes
print(fecha.strftime('%A %d de %B, %H:%M'))  # nombres en inglés
print(fecha.strftime('Día %j del año'))      # día del año

> Los nombres de meses y días (`%B`, `%A`) salen en inglés porque dependen de la configuración regional del sistema, que en Colab es inglés.

## 6. Aritmética con fechas: `timedelta`

| Operación | Resultado | Ejemplo |
|---|---|---|
| fecha − fecha | `timedelta` (duración) | `date(2025,3,10) - date(2025,2,27)` → 11 días |
| fecha ± `timedelta` | fecha | `date(2025,2,27) + timedelta(days=10)` |
| fecha + fecha | **no permitido** (`TypeError`) | Sumar dos instantes no tiene significado |

Un `timedelta` guarda la duración en días, segundos y microsegundos. Dos formas de leerla:

- `.days`: solo los días completos.
- `.total_seconds()`: la duración total en segundos (para convertir a minutos u horas).

In [ ]:
# Resta de fechas → timedelta
fecha1 = dt.date(2025, 3, 10)
fecha2 = dt.date(2025, 2, 27)
diferencia = fecha1 - fecha2
print('Diferencia:', diferencia, '| tipo:', type(diferencia).__name__)
print('Días      :', diferencia.days)

# Resta de fecha-hora → timedelta con horas
salida = dt.datetime(2025, 3, 10, 18, 30)
entrada = dt.datetime(2025, 3, 10, 14, 15)
jornada = salida - entrada
print('\nJornada   :', jornada)
print('Segundos  :', jornada.total_seconds())
print('Horas     :', jornada.total_seconds() / 3600)

**Pregunta**: ¿cuántos días hay entre el 12 de julio de 2018 y el 10 de junio de 2019 a las 5:55? ¿Qué signo tendrá el resultado si restamos la fecha mayor a la menor?

In [ ]:
t4 = dt.datetime(2018, 7, 12, 7, 9, 33)
t5 = dt.datetime(2019, 6, 10, 5, 55, 13)

# Fecha menor - fecha mayor: la duración es negativa
t6 = t4 - t5
print('t4 - t5 =', t6)
print('Días    =', t6.days)

# Fecha mayor - fecha menor: positiva
print('t5 - t4 =', t5 - t4)

**Observación**: `t4 - t5` se muestra como `-333 days, 1:14:20`, mientras que `t5 - t4` es `332 days, 22:45:40`. Python expresa las duraciones negativas como "días negativos más un resto positivo": −333 días + 1:14:20 = −(332 días, 22:45:40). En limpieza de datos, una **duración negativa** (fin antes del inicio) casi siempre indica un error de captura.

### 6.1 Sumar o restar intervalos con `timedelta`

In [ ]:
fecha_base = dt.date(2025, 2, 27)

# Sumar y restar días o semanas
print('+10 días   :', fecha_base + dt.timedelta(days=10))
print('-2 semanas :', fecha_base - dt.timedelta(weeks=2))

# Sumar horas y minutos a una fecha-hora
inicio = dt.datetime(2025, 2, 27, 14, 30)
print('+3 h 30 min:', inicio + dt.timedelta(hours=3, minutes=30))

**Observación**: sumar 10 días al 27 de febrero da el 9 de marzo; `timedelta` toma en cuenta la longitud de cada mes (y los años bisiestos).

## 7. Datos temporales en pandas

pandas tiene sus propios tipos, equivalentes a los de `datetime` pero diseñados para columnas completas:

| `datetime` (Python) | pandas | Uso |
|---|---|---|
| `datetime` | `pd.Timestamp` | Un instante |
| — | columna `datetime64` | Una columna de instantes |
| `timedelta` | `pd.Timedelta` / columna `timedelta64` | Duraciones |
| — | `pd.Period` / columna `period` | Un periodo completo (un mes, un trimestre), no un instante |
| `None` | `NaT` (*Not a Time*) | Valor temporal faltante |

Funciones principales:

| Función | Uso | Sección |
|---|---|---|
| `pd.to_datetime()` | Convertir texto o números a fechas | 7 |
| `pd.to_timedelta()` | Convertir texto o números a duraciones | 9 |
| `pd.date_range()` | Generar una secuencia de fechas | 12 |

> Según la versión de pandas, el tipo aparece como `datetime64[ns]` (nanosegundos) o `datetime64[us]` (microsegundos). Para este curso es equivalente.

### 7.1 Conectar Colab con Google Drive

Ejecuta la siguiente celda y autoriza el acceso. Los archivos deben estar en `MyDrive/Unidad02/`.

In [ ]:
# Monta Google Drive en Colab (solicita autorización la primera vez)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import pandas as pd

# Carpeta donde están los archivos de la sesión
RUTA = '/content/drive/MyDrive/Unidad02/'

### 7.2 Al leer, las fechas son texto

**Pregunta**: `temporal01.csv` contiene las columnas `FechaInicio` y `FechaFin` con valores como `2024-01-05`. ¿De qué tipo serán al leer el archivo?

In [ ]:
# Lectura sin indicaciones especiales
df = pd.read_csv(RUTA + 'temporal01.csv')

print(df)
print('\nTipos de dato:')
print(df.dtypes)

**Observación**: `FechaInicio` y `FechaFin` son texto (`object` o `str`, según la versión). pandas no convierte fechas automáticamente. Hay dos formas de hacerlo.

### 7.3 Convertir al leer: `parse_dates`

```python
pd.read_csv(archivo, parse_dates=['col1', 'col2'])
```

`parse_dates` recibe la **lista** de columnas que deben leerse como fecha.

In [ ]:
# Las columnas indicadas se convierten a datetime64 durante la lectura
df_parse = pd.read_csv(RUTA + 'temporal01.csv', parse_dates=['FechaInicio', 'FechaFin'])

print(df_parse.dtypes)

### 7.4 Convertir después de leer: `pd.to_datetime()`

Ya lo usamos en la sesión de duplicados. Es la opción más flexible, porque permite indicar el formato y decidir qué hacer con los valores que no se pueden convertir.

```python
pd.to_datetime(serie, format=None, errors='raise', dayfirst=False)
```

| Parámetro | Valores | Significado |
|---|---|---|
| `format` | `None` | pandas infiere el formato a partir del **primer valor no nulo** |
| | `'%d-%m-%Y'`, … | Formato exacto (mismos códigos que `strptime`) |
| | `'ISO8601'` | Variantes del estándar ISO (`2025-03-01`, `2025-03-01 14:45`, …) |
| | `'mixed'` | Infiere el formato **valor por valor** (sección 10) |
| `errors` | `'raise'` (por defecto) | Detiene la conversión con un error si un valor no se puede convertir |
| | `'coerce'` | Convierte los valores inválidos en `NaT` |
| `dayfirst` | `False` / `True` | En fechas ambiguas, interpreta el primer número como día |

In [ ]:
df = pd.read_csv(RUTA + 'temporal01.csv')

# Conversión de cada columna de texto a datetime64
df['FechaInicio'] = pd.to_datetime(df['FechaInicio'])
df['FechaFin'] = pd.to_datetime(df['FechaFin'])

print(df.dtypes)

### 7.5 Formato explícito

Cuando el formato no es estándar, indicarlo evita errores y ambigüedades, y además hace la conversión más rápida en archivos grandes.

In [ ]:
ventas = pd.DataFrame({
    'id_venta':    [1, 2, 3],
    'fecha_venta': ['01-03-2025', '02-03-2025', '15-03-2025']   # día-mes-año
})

# Formato explícito: %d = día, %m = mes, %Y = año
ventas['fecha_venta'] = pd.to_datetime(ventas['fecha_venta'], format='%d-%m-%Y')
print(ventas)

### 7.6 Otras fuentes de fechas

**Marca de tiempo Unix** (sección 4): con `unit='s'` pandas interpreta números como segundos desde 1970 (en UTC).

**Componentes en columnas separadas**: si el año, mes y día vienen en columnas distintas, `pd.to_datetime()` puede ensamblarlas, siempre que las columnas se llamen `year`, `month` y `day` (opcionalmente `hour`, `minute`, `second`).

In [ ]:
# Segundos desde 1970-01-01 UTC → fechas
registros = pd.Series([1326244364, 1740765600])
print(pd.to_datetime(registros, unit='s'))

# Fecha armada a partir de columnas separadas
partes = pd.DataFrame({
    'year':  [2025, 2025, 2024],
    'month': [3, 7, 2],
    'day':   [1, 15, 29]
})
partes['fecha'] = pd.to_datetime(partes[['year', 'month', 'day']])
print(partes)

## 8. Extraer componentes: el accesor `.dt`

Así como `.str` aplica métodos de texto a cada valor de una columna (sesión de duplicados), **`.dt`** aplica atributos y métodos de fecha a cada valor de una columna `datetime64`:

| Expresión | Devuelve |
|---|---|
| `col.dt.year`, `.dt.month`, `.dt.day` | Año, mes, día |
| `col.dt.hour`, `.dt.minute` | Hora, minuto |
| `col.dt.dayofweek` | Día de la semana, lunes = 0 (como `weekday()`) |
| `col.dt.day_name()` | Nombre del día (en inglés) |
| `col.dt.month_name()` | Nombre del mes (en inglés) |
| `col.dt.quarter` | Trimestre (1–4) |
| `col.dt.date` | Solo la fecha, sin hora |
| `col.dt.strftime(formato)` | Texto con el formato indicado |

> `.dt` solo funciona sobre columnas de tipo fecha. Sobre una columna de texto produce `AttributeError: Can only use .dt accessor with datetimelike values`: es la señal de que falta convertir con `pd.to_datetime()`.

In [ ]:
# Agregamos columnas con componentes de FechaFin
componentes = df[['Producto', 'FechaFin']].copy()
componentes['anio']       = componentes['FechaFin'].dt.year
componentes['mes']        = componentes['FechaFin'].dt.month
componentes['dia_semana'] = componentes['FechaFin'].dt.dayofweek
componentes['nombre_dia'] = componentes['FechaFin'].dt.day_name()
componentes['trimestre']  = componentes['FechaFin'].dt.quarter
componentes['anio_mes']   = componentes['FechaFin'].dt.strftime('%Y-%m')

print(componentes)

**Observación**: los componentes permiten **agrupar**. Por ejemplo, unidades vendidas según el año en que terminó cada periodo:

In [ ]:
# Unidades por año de FechaFin
print(df.groupby(df['FechaFin'].dt.year)['Unidades'].sum())

> `groupby()` acepta directamente una `Series` como clave (aquí, el año de cada fila) sin necesidad de crear antes una columna.

### 8.1 Periodos: `.dt.to_period()`

Un `Timestamp` es un **instante** (`2025-02-05 00:00:00`). Un `Period` es un **intervalo completo**: el mes de febrero de 2025, el primer trimestre de 2025. Es la forma natural de agrupar por mes o trimestre **sin perder el año** (agrupar solo por `.dt.month` mezcla febrero de 2024 con febrero de 2025).

```python
col.dt.to_period('M')   # frecuencias: 'D' día, 'W' semana, 'M' mes, 'Q' trimestre, 'Y' año
```

**Pregunta**: `FechaInicio` tiene fechas de enero, febrero, marzo y mayo de 2024, y de enero de 2025. ¿Cuántos grupos habrá al agrupar por mes con `.dt.month` y cuántos con `.dt.to_period('M')`?

In [ ]:
periodos = df[['Producto', 'FechaInicio', 'Unidades']].copy()

# Periodo mensual y trimestral de cada fecha
periodos['mes_periodo'] = periodos['FechaInicio'].dt.to_period('M')
periodos['trimestre_periodo'] = periodos['FechaInicio'].dt.to_period('Q')
print(periodos)
print('\nTipo:', periodos['mes_periodo'].dtype)

# Agrupar por número de mes: mezcla años distintos
print('\nUnidades por .dt.month:')
print(periodos.groupby(periodos['FechaInicio'].dt.month)['Unidades'].sum())

# Agrupar por periodo: cada mes de cada año por separado
print('\nUnidades por periodo mensual:')
print(periodos.groupby('mes_periodo')['Unidades'].sum())

**Observación**: con `.dt.month` hay 4 grupos: el mes `1` suma las unidades de enero de 2024 (10) y de enero de 2025 (15), dos periodos distintos. Con `to_period('M')` hay 5 grupos y cada uno indica su año (`2024-01`, `2025-01`). El periodo también se ordena cronológicamente. En la sección 13 lo usaremos con datos reales.

## 9. Operaciones y filtrado con fechas en pandas

### 9.1 Duraciones entre columnas

Restar dos columnas `datetime64` produce una columna `timedelta64`. Sus componentes se leen con `.dt`:

| Expresión | Devuelve |
|---|---|
| `duracion.dt.days` | Días completos |
| `duracion.dt.total_seconds()` | Duración total en segundos |

**Pregunta**: ¿qué tipo de dato tendrá `FechaFin - FechaInicio`?

In [ ]:
df['Duracion'] = df['FechaFin'] - df['FechaInicio']

# Conversión de la duración a distintas unidades
df['Duracion_dias']  = df['Duracion'].dt.days
df['Duracion_horas'] = df['Duracion'].dt.total_seconds() / 3600
df['Duracion_meses'] = df['Duracion'].dt.days / 30.44     # aproximación: días promedio por mes

print(df[['Producto', 'FechaInicio', 'FechaFin', 'Duracion', 'Duracion_dias', 'Duracion_horas', 'Duracion_meses']])
print('\nTipo de Duracion:', df['Duracion'].dtype)

También se puede restar un **solo valor** a toda la columna, por ejemplo la fecha más antigua:

In [ ]:
# Días transcurridos desde el inicio más antiguo
primer_inicio = df['FechaInicio'].min()
print('Primer inicio:', primer_inicio)
print((df['FechaInicio'] - primer_inicio).dt.days)

### 9.2 Sumar intervalos: `pd.Timedelta()`

Equivale a `dt.timedelta`, pero para pandas:

```python
pd.Timedelta(days=3, hours=5)
```

Sumar dos fechas no está permitido; sumar una fecha y un `pd.Timedelta` sí.

In [ ]:
# Entrega estimada: 3 días y 5 horas después de FechaFin
df['FechaEntrega'] = df['FechaFin'] + pd.Timedelta(days=3, hours=5)
print(df[['FechaFin', 'FechaEntrega']])

### 9.3 Convertir texto o números en duraciones: `pd.to_timedelta()`

Así como `pd.to_datetime()` convierte texto en fechas, `pd.to_timedelta()` convierte texto o números en **duraciones**. Es útil cuando un archivo trae tiempos como `"1:30:00"`, `"90 min"` o una columna numérica de minutos.

```python
pd.to_timedelta(serie)             # texto: '1 days 02:00:00', '01:30:00', '90min', '45s'
pd.to_timedelta(serie, unit='m')   # números, indicando la unidad: 's', 'm' (minutos), 'h', 'D'
```

In [ ]:
servicios = pd.DataFrame({
    'orden':      [1, 2, 3, 4],
    'inicio':     pd.to_datetime(['2025-03-01 08:00', '2025-03-01 09:15', '2025-03-01 13:40', '2025-03-01 16:05']),
    'tiempo_txt': ['01:30:00', '45min', '2h', '1 days 00:30:00'],   # duraciones escritas como texto
    'espera_min': [10, 25, 5, 40]                                   # duraciones como número de minutos
})

# Texto → duración
servicios['tiempo'] = pd.to_timedelta(servicios['tiempo_txt'])

# Números → duración, indicando que son minutos
servicios['espera'] = pd.to_timedelta(servicios['espera_min'], unit='m')

# Con duraciones ya se puede calcular la hora de término
servicios['fin'] = servicios['inicio'] + servicios['espera'] + servicios['tiempo']

print(servicios[['orden', 'inicio', 'tiempo_txt', 'tiempo', 'espera', 'fin']])

**Observación**: la orden 4 termina al día siguiente: `1 days 00:30:00` más 40 minutos de espera. Sumar fecha + duraciones es válido; lo que no se puede es sumar dos fechas.

### 9.4 Filtrar por fecha

Una columna `datetime64` se puede comparar con `>`, `<`, `>=`, `<=`, usando texto en formato ISO (`'AAAA-MM-DD'`) o un `pd.Timestamp`. Para un rango cerrado se usa `.between(inicio, fin)`, que incluye ambos extremos.

**Pregunta**: ¿qué filas tienen `FechaFin` posterior al 1 de enero de 2025?

In [ ]:
# Periodos que terminan después del 1 de enero de 2025
print(df[df['FechaFin'] > '2025-01-01'][['Producto', 'FechaInicio', 'FechaFin']])

# Periodos que iniciaron en el primer semestre de 2024 (ambos extremos incluidos)
print(df[df['FechaInicio'].between('2024-01-01', '2024-06-30')][['Producto', 'FechaInicio']])

### 9.5 Ordenar cronológicamente

Con columnas `datetime64`, `sort_values()` ordena por tiempo real (en la sesión de duplicados vimos que el texto se ordena carácter por carácter).

In [ ]:
print(df.sort_values(by='FechaInicio')[['Producto', 'FechaInicio', 'FechaFin']])

## 10. Limpieza de fechas con problemas

El archivo `temporal02.csv` contiene:

```
id_venta,fecha_venta,total_venta
1,2025/03/01,150.50
2,2025-03-02 14:45,200.00
3,March 3 2025,320.75
4,2025-03-04,400.00
5,,500.00
6,2025-03-02 14:45,200.00
```

Identifica los problemas antes de seguir:

- **Formatos mezclados**: `2025/03/01`, `2025-03-02 14:45`, `March 3 2025`, `2025-03-04`.
- Un **faltante** (venta 5).
- Las ventas 2 y 6 tienen la misma fecha, hora y monto: posible **duplicado**.

In [ ]:
# Leemos el archivo usando id_venta como índice
ventas = pd.read_csv(RUTA + 'temporal02.csv', index_col='id_venta')
print(ventas)
print('\n', ventas.dtypes)

### 10.1 Conversión directa

**Pregunta**: ¿qué ocurre al aplicar `pd.to_datetime()` sin más parámetros a esta columna?

> ⚠️ La siguiente celda produce un **error intencional**.

In [ ]:
# ERROR INTENCIONAL: la columna mezcla formatos
pd.to_datetime(ventas['fecha_venta'])

**Explicación del error**: `ValueError: time data "2025-03-02 14:45" doesn't match format "%Y/%m/%d"`.

Desde pandas 2.0, `pd.to_datetime()` **infiere el formato a partir del primer valor no nulo** (`2025/03/01` → `%Y/%m/%d`) y exige que todos los demás lo cumplan. El segundo valor usa guiones y hora, así que la conversión se detiene. El mensaje del error sugiere las alternativas.

### 10.2 `errors='coerce'`: cuidado con lo que se pierde

`errors='coerce'` evita el error convirtiendo en `NaT` lo que no se puede interpretar.

**Pregunta**: ¿cuántos valores quedarán como `NaT`?

In [ ]:
# coerce: lo que no cumple el formato inferido se convierte en NaT
convertidas = pd.to_datetime(ventas['fecha_venta'], errors='coerce')
print(convertidas)
print('\nNaT:', convertidas.isna().sum(), 'de', len(convertidas))

**Observación**: ¡5 de 6 valores quedaron como `NaT`! Solo había un faltante real (venta 5); las otras 4 fechas eran válidas, pero no cumplían el formato inferido del primer valor.

`errors='coerce'` no "arregla" los datos: **oculta** el problema. Siempre compara el número de `NaT` **antes** y **después** de convertir:

In [ ]:
# Faltantes originales vs. faltantes después de convertir
print('Faltantes en el texto original:', ventas['fecha_venta'].isna().sum())
print('NaT después de coerce         :', convertidas.isna().sum())

Si el segundo número es mayor que el primero, la conversión **perdió** fechas válidas.

### 10.3 `format='mixed'`: formato valor por valor

Con `format='mixed'`, pandas interpreta cada valor por separado.

In [ ]:
ventas['fecha_venta'] = pd.to_datetime(ventas['fecha_venta'], format='mixed')
print(ventas)
print('\nNaT:', ventas['fecha_venta'].isna().sum())
print('Tipo:', ventas['fecha_venta'].dtype)

**Observación**: ahora solo queda como `NaT` la venta 5, que era el faltante real. `March 3 2025` se interpretó correctamente.

`format='mixed'` es cómodo, pero tiene un riesgo: al adivinar valor por valor, puede interpretar mal fechas **ambiguas**.

### 10.4 Ambigüedad día/mes

**Pregunta**: `01/03/2025`, ¿es 1 de marzo o 3 de enero?

In [ ]:
ambigua = pd.Series(['01/03/2025', '15/03/2025'])

# Sin indicación: pandas supone mes/día (convención de EE. UU.) cuando puede
print('Por defecto  :', pd.to_datetime(ambigua, format='mixed').tolist())

# dayfirst=True: el primer número es el día (convención de México)
print('dayfirst=True:', pd.to_datetime(ambigua, format='mixed', dayfirst=True).tolist())

**Observación**: por defecto, `01/03/2025` se interpretó como **3 de enero**, mientras que `15/03/2025` se interpretó como 15 de marzo porque no hay mes 15. ¡La misma columna quedó con dos convenciones distintas sin ningún aviso! Con `dayfirst=True` ambas son de marzo.

**Regla práctica**: si conoces el formato, **indícalo** con `format`. Usa `'mixed'` solo cuando los formatos realmente varían, y revisa el resultado.

### 10.5 Fechas imposibles

`errors='coerce'` sí es útil para detectar valores que **no pueden** ser fechas: se convierten en `NaT` y se pueden aislar para revisarlos.

In [ ]:
capturas = pd.DataFrame({
    'folio': [101, 102, 103, 104],
    'fecha': ['2025-02-28', '2025-02-30', '2025-13-01', '2025-03-15']
})

# Formato explícito + coerce: lo imposible se convierte en NaT
capturas['fecha_dt'] = pd.to_datetime(capturas['fecha'], format='%Y-%m-%d', errors='coerce')

# Filas que no se pudieron convertir, con su texto original para revisión
print(capturas[capturas['fecha_dt'].isna()])

**Observación**: el 30 de febrero y el mes 13 quedaron aislados. Conservar la columna de texto original permite revisar **por qué** falló cada valor.

### 10.6 Revisión final: faltantes y duplicados

Volvemos a `ventas`. Con las fechas ya convertidas, podemos aplicar lo visto en la sesión de duplicados.

In [ ]:
# Faltantes de fecha
print('Ventas sin fecha:', ventas['fecha_venta'].isna().sum())

# Ventas repetidas: misma fecha-hora y mismo monto (el índice id_venta no se compara)
repetidas = ventas.duplicated(keep=False)
print('\nVentas repetidas:')
print(ventas[repetidas])

**Observación**: las ventas 2 y 6 coinciden en fecha, hora y monto. Con solo estas columnas no se puede saber si son la misma venta capturada dos veces o dos ventas reales en el mismo minuto; se registran para revisión. Antes de la conversión, `duplicated()` también las habría detectado porque el texto era idéntico; pero en general, **los duplicados se detectan mejor después de normalizar las fechas** (`'2025-03-02 14:45'` y `'2025/03/02 14:45'` son textos distintos y fechas iguales).

## 11. Zonas horarias en pandas

Las columnas `datetime64` también pueden ser *naive* o *aware*. Se trabajan con dos métodos, equivalentes a los de la sección 3:

| Método | Equivalente en `datetime` | Efecto |
|---|---|---|
| `col.dt.tz_localize(zona)` | `replace(tzinfo=...)` | Asigna la zona; no cambia la hora |
| `col.dt.tz_convert(zona)` | `astimezone(...)` | Convierte a otra zona; cambia la hora |

In [ ]:
# Registros de un servidor, guardados en UTC (naive)
registros = pd.DataFrame({
    'evento': ['inicio de sesión', 'compra', 'cierre de sesión'],
    'hora_utc': pd.to_datetime(['2025-07-10 15:00', '2025-07-10 15:20', '2025-12-10 15:45'])
})

# 1) Declarar que están en UTC
registros['hora_utc'] = registros['hora_utc'].dt.tz_localize('UTC')

# 2) Convertir a hora de Ciudad Juárez
registros['hora_juarez'] = registros['hora_utc'].dt.tz_convert('America/Ciudad_Juarez')

print(registros)

**Observación**: en julio la diferencia es de 6 horas (horario de verano) y en diciembre de 7.

El horario de verano también produce horas que **no existen**: el segundo domingo de marzo, en Juárez, el reloj salta de 01:59 a 03:00.

**Pregunta**: ¿qué pasa si intentamos localizar las 02:30 del 9 de marzo de 2025 en Juárez?

> ⚠️ La siguiente celda produce un **error intencional**.

In [ ]:
# ERROR INTENCIONAL: 2025-03-09 02:30 no existe en Ciudad Juárez (cambio de horario)
hora_local = pd.Series(pd.to_datetime(['2025-03-09 01:30', '2025-03-09 02:30']))
hora_local.dt.tz_localize('America/Ciudad_Juarez')

**Explicación del error**: `ValueError: 2025-03-09 02:30:00 is a nonexistent time due to daylight savings time`.

Una hora local entre 02:00 y 02:59 de ese día indica un error de captura o un reloj mal configurado. El parámetro `nonexistent` permite decidir qué hacer, por ejemplo convertirla en `NaT` para revisarla:

In [ ]:
# nonexistent='NaT': las horas inexistentes se marcan como faltantes
print(hora_local.dt.tz_localize('America/Ciudad_Juarez', nonexistent='NaT'))

> **Buena práctica**: almacenar en UTC y convertir a hora local solo para presentar o para analizar patrones por hora del día.

## 12. Caso real 1: calidad del aire (Airdata, 2020)

`Airdata.csv` contiene mediciones **horarias** durante 2020: temperatura, humedad, velocidad y dirección del viento, y concentración de dióxido de nitrógeno (NO₂, en µg/m³) en tres ubicaciones (A, B y C). El NO₂ proviene principalmente de la combustión de vehículos, por lo que se espera que siga los patrones del tráfico.

Aplicaremos el flujo completo: conversión, validación de la línea de tiempo, extracción de componentes y análisis por periodos.

### 12.1 Carga y conversión

In [ ]:
aire = pd.read_csv(RUTA + 'Airdata.csv')

print('Dimensiones:', aire.shape)
print(aire.head())
print('\nTipo de DateTime:', aire['DateTime'].dtype)

La columna `DateTime` tiene valores como `1/1/2020 0:00` y `1/13/2020 12:00`. El formato es **mes/día/año hora:minuto**. Una fecha como `1/2/2020` es ambigua (¿2 de enero o 1 de febrero?); `1/13/2020` confirma que el primer número es el mes.

Indicamos el formato explícitamente: `%m/%d/%Y %H:%M`. Los códigos aceptan valores sin cero inicial (`1` en lugar de `01`).

In [ ]:
# Formato explícito: mes/día/año hora:minuto
aire['DateTime'] = pd.to_datetime(aire['DateTime'], format='%m/%d/%Y %H:%M')

print(aire['DateTime'].dtype)
print('Primera medición:', aire['DateTime'].min())
print('Última medición :', aire['DateTime'].max())

### 12.2 Validar la línea de tiempo

En datos horarios hay que confirmar tres cosas:

1. **Completitud**: que no falte ninguna hora.
2. **Unicidad**: que ninguna hora aparezca dos veces.
3. **Regularidad**: que el intervalo entre mediciones sea constante.

### Función: `pd.date_range()`

```python
pd.date_range(start=inicio, end=fin, freq='h')
```

Genera **todas** las fechas entre `start` y `end` con la frecuencia indicada: `'h'` (hora), `'D'` (día), `'min'` (minuto), `'MS'` (inicio de mes). Sirve como "calendario de referencia" para compararlo con los datos.

**Pregunta**: 2020 fue año bisiesto. ¿Cuántas horas tiene?

In [ ]:
# Calendario de referencia: todas las horas entre la primera y la última medición
calendario = pd.date_range(start=aire['DateTime'].min(), end=aire['DateTime'].max(), freq='h')
print('Horas esperadas :', len(calendario))
print('Filas en datos  :', len(aire))

# 1) Completitud: horas del calendario que NO aparecen en los datos
faltantes = calendario[~calendario.isin(aire['DateTime'])]
print('Horas faltantes :', len(faltantes))

# 2) Unicidad: horas repetidas
print('Horas repetidas :', aire['DateTime'].duplicated().sum())

**Observación**: 366 días × 24 horas = 8,784 horas, exactamente las filas del archivo, sin faltantes ni repetidas.

### Método: `Series.diff()`

`serie.diff()` calcula la diferencia entre cada valor y el **anterior**. Sobre una columna de fechas devuelve un `timedelta64`: el tiempo transcurrido entre registros consecutivos. El primer valor es `NaT` porque no tiene anterior.

In [ ]:
# 3) Regularidad: intervalo entre mediciones consecutivas
intervalos = aire['DateTime'].diff()
print(intervalos.head())

# ¿Qué intervalos aparecen y cuántas veces?
print('\nIntervalos distintos:')
print(intervalos.value_counts())

**Observación**: los 8,783 intervalos son de exactamente 1 hora: la línea de tiempo es regular.

> Si hubiera intervalos de 2 horas o más, `diff()` los revelaría como huecos; intervalos de 0 indicarían horas repetidas. Nota también que el archivo **no** tiene el salto del horario de verano: probablemente las horas están en una zona sin cambio de horario o en UTC. Es un supuesto que convendría confirmar con quien generó los datos.

### 12.3 Faltantes en el tiempo

La línea de tiempo está completa, pero algunas **mediciones** de NO₂ faltan. Veamos cómo se distribuyen por mes.

In [ ]:
columnas_no2 = ['NO2_Location_A', 'NO2_Location_B', 'NO2_Location_C']

# Faltantes totales por ubicación
print('Faltantes por ubicación:')
print(aire[columnas_no2].isna().sum())

# Faltantes por mes: isna() da True/False; sum() cuenta los True dentro de cada mes
print('\nFaltantes por mes:')
print(aire[columnas_no2].isna().groupby(aire['DateTime'].dt.month).sum())

**Observación**: la ubicación B tiene entre 4 y 5 veces más faltantes que A o C (580 horas, ≈ 6.6 % del año), con más ausencias en febrero, marzo y diciembre. La ubicación C tiene exactamente 11 faltantes cada mes, un patrón tan regular que sugiere una causa sistemática (por ejemplo, mantenimiento programado). Estos patrones solo se ven al analizar los faltantes **en el tiempo**.

### 12.4 Extraer componentes

Creamos columnas con los componentes que usaremos para agrupar.

In [ ]:
aire['mes']        = aire['DateTime'].dt.month
aire['hora']       = aire['DateTime'].dt.hour
aire['dia_semana'] = aire['DateTime'].dt.dayofweek          # lunes = 0
aire['fin_semana'] = aire['dia_semana'] >= 5                # sábado (5) y domingo (6)

print(aire[['DateTime', 'mes', 'hora', 'dia_semana', 'fin_semana']].head())

### 12.5 Patrones temporales

**Pregunta**: si el NO₂ proviene del tráfico, ¿será mayor entre semana o en fin de semana? ¿A qué horas del día esperas los valores más altos?

In [ ]:
# Promedio de NO2: entre semana vs. fin de semana
print('Promedio de NO2 (False = entre semana, True = fin de semana):')
print(aire.groupby('fin_semana')[columnas_no2].mean().round(1))

In [ ]:
# Perfil horario: promedio de NO2 para cada hora del día
perfil_horario = aire.groupby('hora')[columnas_no2].mean().round(1)
print(perfil_horario)

**Observación**: el NO₂ es entre 16 % y 24 % menor en fin de semana, y el perfil horario muestra un máximo en la mañana (entre las 7:00 y las 8:00, horario de entrada al trabajo y a la escuela) y mínimos de madrugada (alrededor de las 3:00). Ambos patrones son coherentes con el tráfico vehicular.

> `mean()` ignora los `NaN` por defecto: cada promedio se calcula con las horas que sí tienen medición.

### Función: `plt.plot()`

En la sesión de análisis descriptivo usamos `plt.hist()` y `plt.boxplot()`. Para mostrar cómo cambia un valor a lo largo de una secuencia ordenada (las 24 horas del día) se usa una gráfica de **líneas**:

```python
plt.plot(x, y, label='nombre')   # una línea; label se usa en la leyenda
plt.legend()                     # muestra la leyenda con los label
```

Se pueden dibujar varias líneas en la misma gráfica llamando `plt.plot()` varias veces antes de `plt.show()`.

In [ ]:
import matplotlib.pyplot as plt

# Una línea por ubicación: eje x = hora del día, eje y = NO2 promedio
for columna in columnas_no2:
    plt.plot(perfil_horario.index, perfil_horario[columna], marker='o', label=columna)

plt.title('Perfil horario promedio de NO$_2$, 2020')
plt.xlabel('Hora del día')
plt.ylabel('NO$_2$ promedio (µg/m³)')
plt.xticks(range(0, 24, 2))
plt.legend()
plt.show()

### 12.6 Análisis por meses y filtrado por periodo

In [ ]:
# Promedio mensual de NO2 por ubicación
promedio_mensual = aire.groupby('mes')[columnas_no2].mean().round(1)
print(promedio_mensual)

**Observación**: enero registra los valores más altos en B y C (en A, noviembre supera ligeramente a enero), y mayo es el mes más bajo en las tres ubicaciones. Varios factores pueden explicarlo (calefacción y menor dispersión atmosférica en invierno), y en 2020 también coincide con el confinamiento por COVID-19 en muchos países. Con estos datos solo podemos **describir** el patrón, no atribuirle una causa.

Para analizar un periodo específico se filtra con `between()`:

In [ ]:
# Periodo de marzo a mayo de 2020 (hasta la última hora del 31 de mayo)
primavera = aire[aire['DateTime'].between('2020-03-01', '2020-05-31 23:00')]
print('Horas en el periodo:', len(primavera))
print(primavera[columnas_no2].mean().round(1))

# Horas pico de la mañana (7:00 a 9:00) en días hábiles
pico = aire[aire['hora'].between(7, 9) & ~aire['fin_semana']]
print('\nHoras pico en días hábiles:', len(pico))
print(pico[columnas_no2].mean().round(1))

### 12.7 La fecha como índice: vista previa

En la sección 1.2 comparamos la fecha como **columna** y como **índice**. Hasta ahora usamos la columna. Veamos qué cambia al usarla como índice.

### Método: `DataFrame.set_index()`

```python
df.set_index('columna')
```

Devuelve un nuevo `DataFrame` en el que la columna indicada pasa a ser el índice. Si la columna es de fechas, el índice es un `DatetimeIndex`, que permite:

| Operación | Código | Resultado |
|---|---|---|
| Seleccionar un periodo con texto | `serie.loc['2020-03']` | Todas las filas de marzo de 2020 |
| Seleccionar un rango | `serie.loc['2020-03-01':'2020-03-15']` | Del 1 al 15 de marzo, ambos incluidos |
| Cambiar la frecuencia | `serie.resample('D').mean()` | Promedio por día (de horario a diario) |
| Ventana deslizante | `serie.rolling(24).mean()` | Promedio de las últimas 24 filas en cada punto |

In [ ]:
# La fecha pasa a ser el índice (la hora ya es única, requisito de un buen índice)
aire_idx = aire.set_index('DateTime')
print(type(aire_idx.index).__name__)

# Selección por periodo usando texto
marzo = aire_idx.loc['2020-03', columnas_no2]
print('Horas en marzo de 2020:', len(marzo))

# Selección por rango de fechas
quincena = aire_idx.loc['2020-03-01':'2020-03-15', columnas_no2]
print('Horas del 1 al 15 de marzo:', len(quincena))

### Método: `resample()`

`serie.resample(frecuencia).función()` agrupa los registros en intervalos de tiempo (`'D'` día, `'W'` semana, `'MS'` mes) y aplica una función de resumen. Es el equivalente temporal de `groupby()`, pero requiere que la fecha sea el índice.

### Método: `rolling()`

`serie.rolling(n).mean()` calcula, para cada fila, el promedio de las últimas `n` filas: una **media móvil**. Con datos horarios, `rolling(24)` suaviza las variaciones dentro del día.

In [ ]:
# De datos horarios a promedios diarios
diario = aire_idx['NO2_Location_C'].resample('D').mean()
print('Días:', len(diario))
print(diario.head())

# Media móvil de 24 horas sobre los datos horarios
movil_24h = aire_idx['NO2_Location_C'].rolling(24, min_periods=18).mean()   # min_periods: mínimo de horas con dato
print(movil_24h.loc['2020-01-01 20:00':'2020-01-02 02:00'])

In [ ]:
import matplotlib.pyplot as plt

# Marzo de 2020: datos horarios vs. media móvil de 24 horas en la ubicación C
tramo = aire_idx.loc['2020-03', 'NO2_Location_C']
plt.plot(tramo.index, tramo, linewidth=0.6, label='Horario')
plt.plot(tramo.index, movil_24h.loc['2020-03'], linewidth=2, label='Media móvil 24 h')
plt.title('NO$_2$ en la ubicación C, marzo de 2020')
plt.xlabel('Fecha')
plt.ylabel('NO$_2$ (µg/m³)')
plt.xticks(rotation=45)
plt.legend()
plt.show()

**Observación**: la media móvil elimina el ciclo diario (picos de la mañana, mínimos de madrugada) y deja ver la tendencia de fondo. `resample()` y `rolling()` son herramientas de **series temporales**; aquí solo las presentamos para mostrar por qué, en ese tipo de análisis, la fecha se usa como índice.

> `min_periods=18` indica que el promedio se calcula si al menos 18 de las 24 horas tienen dato; si hay menos, el resultado es `NaN`. Así los faltantes no producen promedios engañosos.

## 13. Caso real 2: fechas en texto con problemas (catálogo de Netflix)

El catálogo de Netflix publicado por TidyTuesday (abril de 2021) contiene 7,787 títulos. La columna `date_added` indica cuándo se agregó cada título a la plataforma, escrita como texto en inglés: `"September 25, 2020"`.

Objetivos:

1. Convertir un texto de fecha con nombre de mes y detectar valores que fallan.
2. Usar periodos (`to_period`) para contar por mes y año.
3. Detectar anomalías comparando dos variables temporales.

In [ ]:
# Descarga directa desde el repositorio de TidyTuesday en GitHub
url_netflix = 'https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2021/2021-04-20/netflix_titles.csv'
netflix = pd.read_csv(url_netflix)

print('Dimensiones:', netflix.shape)
print(netflix[['title', 'type', 'release_year', 'date_added']].head())
print('\nFaltantes en date_added:', netflix['date_added'].isna().sum())

### 13.1 Conversión con formato explícito

El formato es: nombre completo del mes (`%B`), día (`%d`), coma, año (`%Y`) → `'%B %d, %Y'`.

**Pregunta**: la columna tiene 10 faltantes. Si el formato es correcto, ¿cuántos `NaT` esperas después de convertir con `errors='coerce'`?

In [ ]:
formato_netflix = '%B %d, %Y'

intento = pd.to_datetime(netflix['date_added'], format=formato_netflix, errors='coerce')
print('Faltantes antes de convertir:', netflix['date_added'].isna().sum())
print('NaT después de convertir    :', intento.isna().sum())

**Observación**: aparecen 98 `NaT`, no 10. Se perdieron 88 fechas. Aislemos los valores que fallaron **sin ser faltantes** y mostremos su texto con `repr()` para ver los caracteres ocultos:

In [ ]:
# Valores que tenían texto pero no se pudieron convertir
fallidos = netflix.loc[intento.isna() & netflix['date_added'].notna(), 'date_added']
print('Valores fallidos:', len(fallidos))

# repr() muestra las comillas: así se ven los espacios al inicio
print(fallidos.head().map(repr).tolist())

**Observación**: los 88 valores empiezan con un **espacio** (`' August 4, 2017'`), que no coincide con el formato. La solución es la de la sesión de duplicados: `.str.strip()` antes de convertir.

In [ ]:
# Quitamos espacios al inicio y al final, y convertimos
netflix['fecha_agregado'] = pd.to_datetime(netflix['date_added'].str.strip(), format=formato_netflix, errors='coerce')

print('NaT después de strip + convertir:', netflix['fecha_agregado'].isna().sum())
print('Rango:', netflix['fecha_agregado'].min(), '→', netflix['fecha_agregado'].max())

**Observación**: ahora solo quedan los 10 faltantes reales. Sin la comparación antes/después, habríamos perdido 88 títulos en silencio.

### 13.2 Conteo por periodo y por día de la semana

**Pregunta**: ¿en qué día de la semana crees que Netflix agrega más títulos?

In [ ]:
# Títulos agregados por año (periodo anual)
por_anio = netflix['fecha_agregado'].dt.to_period('Y').value_counts().sort_index()
print('Títulos agregados por año:')
print(por_anio)

In [ ]:
# Distribución por día de la semana (en proporción)
print('Proporción por día de la semana:')
print(netflix['fecha_agregado'].dt.day_name().value_counts(normalize=True).round(3))

# ¿Qué proporción se agrega el día 1 del mes?
print('\nProporción agregada el día 1 del mes:', round((netflix['fecha_agregado'].dt.day == 1).mean(), 3))

**Observación**: el viernes concentra cerca del 29 % de los títulos (si fuera uniforme, cada día tendría ≈ 14 %), coherente con la práctica de estrenar los viernes. Además, ≈ 27 % de los títulos se agregan el **día 1 del mes**, cuando un mes tiene 30 o 31 días: son cargas masivas al inicio de cada mes (típicamente, catálogo con licencia que entra en bloque). Estos patrones solo aparecen al extraer componentes de la fecha.

> 2021 aparece con pocos títulos porque los datos terminan el 16 de enero de 2021: un periodo **incompleto**. Compararlo con años completos sería un error.

### 13.3 Anomalías entre dos variables temporales

`release_year` es el año de estreno. Un título no debería agregarse **antes** de estrenarse.

In [ ]:
# Diferencia en años entre la llegada a Netflix y el estreno
netflix['anios_hasta_netflix'] = netflix['fecha_agregado'].dt.year - netflix['release_year']

anomalos = netflix[netflix['anios_hasta_netflix'] < 0]
print('Títulos agregados antes de su año de estreno:', len(anomalos))
print(anomalos[['title', 'type', 'release_year', 'date_added']].head(8))

**Observación**: 12 títulos se agregaron antes de su año de estreno. En series (`TV Show`), `release_year` suele corresponder a la **temporada más reciente**, no a la primera; en películas puede ser un error o un estreno anticipado en la plataforma. Es un ejemplo de por qué la validación temporal requiere **conocer el significado** de cada columna antes de corregir.

## 14. Caso real 3: fechas en UTC y varias estaciones (OpenAQ)

La documentación de pandas incluye mediciones de NO₂ de tres estaciones (París, Amberes y Londres) obtenidas de la plataforma OpenAQ, mayo–junio de 2019. Retos:

1. La fecha viene **con zona horaria** (`+00:00`, UTC).
2. Hay **varias estaciones** en la misma tabla: la validación de la línea de tiempo debe hacerse **por estación**.
3. Los patrones horarios cambian según se lean en UTC o en hora local.

In [ ]:
url_aq = 'https://raw.githubusercontent.com/pandas-dev/pandas/main/doc/data/air_quality_no2_long.csv'
aq = pd.read_csv(url_aq)

print('Dimensiones:', aq.shape)
print(aq.head())

### 14.1 Conversión de fechas con zona horaria

**Pregunta**: el texto termina en `+00:00`. ¿La columna convertida será *naive* o *aware*?

In [ ]:
aq['fecha_utc'] = pd.to_datetime(aq['date.utc'])

print('Tipo:', aq['fecha_utc'].dtype)
print('Rango:', aq['fecha_utc'].min(), '→', aq['fecha_utc'].max())
print('\nMediciones por estación:')
print(aq.groupby(['city', 'location']).size())

**Observación**: pandas reconoce el desfase y crea una columna *aware* en UTC (`datetime64[..., UTC]`). Amberes tiene 95 mediciones, frente a casi 1,000 de las otras estaciones, en el mismo periodo. ¿Faltan datos o mide con otra frecuencia?

### 14.2 Regularidad por estación

En la sección 12 usamos `diff()` para una sola serie. Con varias estaciones, calcularlo sobre toda la columna mezclaría la última medición de una estación con la primera de otra. Hay que calcularlo **dentro de cada grupo**.

### Método: `groupby(...)[col].diff()`

`df.groupby('clave')['fecha'].diff()` calcula la diferencia con la fila anterior **del mismo grupo**. La primera fila de cada grupo queda como `NaT`. Los datos deben estar **ordenados** por grupo y fecha.

In [ ]:
# Ordenar por estación y fecha antes de calcular diferencias
aq = aq.sort_values(by=['location', 'fecha_utc']).reset_index(drop=True)

# Intervalo con la medición anterior de la MISMA estación
aq['intervalo'] = aq.groupby('location')['fecha_utc'].diff()

# Los 3 intervalos más frecuentes de cada estación
for estacion in aq['location'].unique():
    intervalos_est = aq.loc[aq['location'] == estacion, 'intervalo'].value_counts()
    print(estacion)
    print(intervalos_est.head(3), '\n')

# Hueco más largo de cada estación
print('Hueco más largo por estación:')
print(aq.groupby('location')['intervalo'].max())

**Observación**:

- **París** y **Londres** miden cada hora, con huecos ocasionales (el mayor, de poco más de un día en París y 6 horas en Londres).
- **Amberes**: la mayoría de sus intervalos son de 1 hora, pero hay 26 saltos de casi un día (23 o 24 horas) y un hueco de 4 días. Mide en bloques cortos separados por días sin datos: 95 mediciones en 45 días. Es un **muestreo irregular**, no una serie horaria completa.

Sin esta revisión, comparar promedios entre estaciones supondría erróneamente que todas miden igual.

### 14.3 UTC vs. hora local

**Pregunta**: en París, en verano, la hora local es UTC+2. Si el pico de contaminación ocurre a la hora de entrada al trabajo (≈ 8:00–9:00 local), ¿a qué hora aparecerá en UTC?

In [ ]:
paris = aq[aq['city'] == 'Paris'].copy()

# Conversión a la hora local de París (horario de verano: UTC+2)
paris['fecha_local'] = paris['fecha_utc'].dt.tz_convert('Europe/Paris')

# Perfil horario promedio según cada referencia horaria
perfil_utc = paris.groupby(paris['fecha_utc'].dt.hour)['value'].mean().round(1)
perfil_local = paris.groupby(paris['fecha_local'].dt.hour)['value'].mean().round(1)

print('Hora pico en UTC  :', perfil_utc.sort_values().index[-1], 'h')
print('Hora pico en local:', perfil_local.sort_values().index[-1], 'h')

plt.plot(perfil_utc.index, perfil_utc, marker='o', label='Hora UTC')
plt.plot(perfil_local.index, perfil_local, marker='o', label='Hora local (Europe/Paris)')
plt.title('París: NO$_2$ promedio por hora del día, mayo–junio 2019')
plt.xlabel('Hora del día')
plt.ylabel('NO$_2$ (µg/m³)')
plt.xticks(range(0, 24, 2))
plt.legend()
plt.show()

**Observación**: el pico aparece a las 7:00 en UTC y a las 9:00 en hora local. Es la misma curva desplazada 2 horas. Si se reportara "el pico ocurre a las 7:00" usando UTC, la conclusión sobre la hora de mayor tráfico sería incorrecta. **Regla**: para analizar comportamiento humano (tráfico, compras, uso de servicios), convierte a la hora local del lugar donde ocurre.

## 15. Cierre

### Resumen

| Tarea | Python (`datetime`) | pandas |
|---|---|---|
| Crear | `dt.datetime(2025, 2, 28, 18, 0)` | `pd.Timestamp('2025-02-28 18:00')` |
| Texto → fecha | `dt.datetime.strptime(texto, '%d/%m/%Y')` | `pd.to_datetime(col, format='%d/%m/%Y')` |
| Convertir al leer | — | `pd.read_csv(archivo, parse_dates=['col'])` |
| Fecha → texto | `fecha.strftime('%Y-%m')` | `col.dt.strftime('%Y-%m')` |
| Componentes | `fecha.year`, `fecha.weekday()` | `col.dt.year`, `col.dt.dayofweek`, `col.dt.day_name()` |
| Duración | `fecha2 - fecha1` → `timedelta` | `col2 - col1` → `timedelta64` |
| Leer duración | `.days`, `.total_seconds()` | `.dt.days`, `.dt.total_seconds()` |
| Sumar intervalo | `fecha + dt.timedelta(days=3)` | `col + pd.Timedelta(days=3)` |
| Filtrar | — | `df[col > '2025-01-01']`, `col.between(a, b)` |
| Faltante | `None` | `NaT`; se detecta con `.isna()` |
| Unix → fecha | `dt.datetime.fromtimestamp(n, tz=dt.timezone.utc)` | `pd.to_datetime(col, unit='s')` |
| Zona horaria | `replace(tzinfo=)`, `astimezone()` | `.dt.tz_localize()`, `.dt.tz_convert()` |
| Periodo (mes, trimestre) | — | `col.dt.to_period('M')` |
| Texto/número → duración | — | `pd.to_timedelta(col)`, `pd.to_timedelta(col, unit='m')` |
| Calendario de referencia | — | `pd.date_range(inicio, fin, freq='h')` |
| Intervalo entre registros | — | `col.diff()`; por grupo: `df.groupby('clave')['fecha'].diff()` |
| Fecha como índice | — | `df.set_index('fecha')`, `.loc['2020-03']` |
| Cambiar frecuencia / media móvil | — | `.resample('D').mean()`, `.rolling(24).mean()` |

### Flujo de limpieza de datos temporales

1. **Revisar el tipo** con `dtypes`: si es texto, hay que convertir.
2. **Inspeccionar los valores**: ¿un formato o varios? ¿día/mes o mes/día? ¿hay hora? ¿zona horaria? ¿espacios ocultos (usa `repr()`)?
3. **Convertir** con `format` explícito; `'mixed'` solo si los formatos realmente varían.
4. **Comparar faltantes** antes y después de convertir: un aumento indica fechas perdidas.
5. **Aislar y revisar** los `NaT` nuevos (fechas imposibles, formatos no previstos).
6. **Validar rangos**: fechas fuera del periodo esperado, duraciones negativas o imposibles.
7. **Validar la línea de tiempo** (si aplica): completitud (`date_range`), unicidad (`duplicated`), regularidad (`diff`), **por grupo** si hay varias series.
8. **Definir la referencia horaria**: UTC para almacenar, hora local para analizar comportamiento.
9. **Revisar duplicados** después de convertir.
10. **Extraer componentes** y analizar; usar periodos para agrupar por mes o año sin mezclar años.

### Buenas prácticas

**Lo que se puede hacer**

- Indicar el formato cuando se conoce: `pd.to_datetime(col, format='%m/%d/%Y %H:%M')`.
- Convertir al leer cuando el formato es estándar: `pd.read_csv(archivo, parse_dates=['fecha'])`.
- Conservar la columna original de texto mientras se valida la conversión: `df['fecha_dt'] = pd.to_datetime(df['fecha'], errors='coerce')`.
- Contar `NaT` antes y después: `df['fecha'].isna().sum()` vs. `df['fecha_dt'].isna().sum()`.
- Usar `dayfirst=True` con fechas en formato mexicano ambiguas: `pd.to_datetime(col, format='mixed', dayfirst=True)`.
- Guardar en UTC y convertir para presentar: `col.dt.tz_localize('UTC').dt.tz_convert('America/Ciudad_Juarez')`.
- Limpiar espacios antes de convertir: `pd.to_datetime(col.str.strip(), format='%B %d, %Y')`.
- Agrupar por periodo para no mezclar años: `df.groupby(col.dt.to_period('M'))`.
- Calcular intervalos por estación o por cliente: `df.sort_values(['clave', 'fecha']).groupby('clave')['fecha'].diff()`.

**Lo que no se puede hacer**

- Usar `.dt` sobre texto: `df['fecha'].dt.month` falla si `fecha` no se convirtió.
- Aplicar `errors='coerce'` sin revisar: en `temporal02.csv` convirtió 4 fechas válidas en `NaT`.
- Confiar en `'mixed'` con fechas ambiguas: `01/03/2025` se interpretó como 3 de enero.
- Sumar dos fechas: `fecha1 + fecha2` produce `TypeError`; solo fecha + duración.
- Confundir `weekday()` (lunes = 0) con `isoweekday()` (lunes = 1).
- Suponer que una serie horaria está completa: verifícalo con `pd.date_range()` y `diff()`.
- Aplicar `diff()` a una columna con varias series mezcladas: compara la última medición de una estación con la primera de otra.
- Agrupar por `.dt.month` con datos de varios años: enero de 2024 y enero de 2025 quedan en el mismo grupo.
- Comparar un periodo incompleto con uno completo: 2021 en Netflix solo tiene 16 días.
- Interpretar horas en UTC como hora local: en París el pico de NO₂ parece ocurrir a las 7:00 cuando ocurre a las 9:00.

---
**Fin del notebook.**